# Figures settings

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import seaborn as sns
import variograd_utils as vu
from variograd_utils.surfplotter import SurfPlotter
from variograd_utils import dataset


wmin = 5
wmax = 14
dpi=900

plt.rcParams['axes.titlesize'] = 14 # Title size
plt.rcParams['axes.labelsize'] = 12   # Axis labels size
plt.rcParams['xtick.labelsize'] = 10   # X-axis tick labels size
plt.rcParams['ytick.labelsize'] = 10   # Y-axis tick labels size
plt.rcParams['legend.fontsize'] = 12    # Legend font size
plt.rcParams['legend.title_fontsize'] = 12    # Legend title font size
plt.rcParams['axes.facecolor'] = "none"
plt.rcParams['figure.facecolor'] = "none"
plt.rcParams['legend.frameon'] = False
plt.rcParams['axes.edgecolor'] = "none"
plt.rcParams['figure.dpi'] = dpi
plt.rcParams['savefig.dpi'] = dpi
plt.rcParams['figure.constrained_layout.use'] = True


def style_axis(ax):
    ax.set_facecolor('none')
    ax.set_frame_on(False)
    return ax

def style_figure(figure=None):
    if figure is None:
        figure = plt.gcf()
    figure.set_facecolor('none')
    figure.set_dpi(dpi)
    for ax in figure.get_axes():
        style_axis(ax)

babylav = "#9962B3"
blusky = "#7DADE0"
daffodil = "#F7E189"
peach = "#FBB28B"
poppy = "#F76F54"
fuchsia = "#EA5E86"
night = "#252D45"
sage = "#A4A48A"
hay = "#B79A65"
candy = "#F9A2C5"
eggplant = "#4A3741"
pool = "#47B5A8"
forest = "#2C2A08"
w = "#F5F5F5"
k = "#242F36"

gradmap = mpl.colors.LinearSegmentedColormap.from_list('gradmap', [babylav, poppy, pool], N=3)

sns.set_palette([poppy, blusky, daffodil, candy, pool, hay, babylav, peach, sage, fuchsia])

cmap_subj = mpl.colors.LinearSegmentedColormap.from_list('subj_cmap', [poppy, blusky, daffodil], N=3)
cmap1 = mpl.colors.LinearSegmentedColormap.from_list('custom_cmap', [k, poppy, daffodil, w], N=256, gamma=1.1)

## Fig 1

In [ ]:
# Set data to plot

h = "L" # hemisphere
s = 50 # kernel size
l = 4 # searchlight side length
g = 0 # gradient index

n_vtx = vu.vertex_info_10k[f"gray{h.lower()}"].size
test = dataset("test")
train = dataset("train")
figpath = train.outpath("figures")
if not os.path.exists(figpath):
    os.makedirs(figpath)

In [ ]:
# Load data
grp = "train"
data  = dataset(grp)

grads = np.loadtxt(data.outpath(f"{grp}.L.FC_embeddings.a05_t0.G{g+1}.csv"), delimiter=",")
grads_avg = grads.mean(axis=0)

gembed_all = data.load_embeddings("L", alg="JE_cauchy")
gembed = gembed_all["JE_cauchy50"][..., :3]
gembed_avg = gembed.mean(axis = 0)

In [ ]:
# FIGURE 1A

from matplotlib.colors import to_rgb

base_colors = np.array([cmap_subj(0), cmap_subj(0.5), cmap_subj(.9)])#

n_regions = 3
n_timepoints = 50

FC = test.outpath("test.REST_FC.10k_fs_LR.npy")
FC = np.load(FC)[vu.vertex_info_10k.grayl][:,vu.vertex_info_10k.grayl]
FC[FC < np.percentile(FC, 95)] = 0

sidx = [0,4,5]
sxyz = [np.loadtxt(data.outpath(f"{grp}.L.FC_embeddings.a05_t0.G{g+1}.csv"), delimiter=",")[sidx] for g in range(3)]
sxyz = np.hstack([i.T.reshape(-1,1) for i in sxyz])
sxyz = np.hstack([np.repeat(np.arange(3), n_vtx).reshape(-1,1), sxyz])
cmark = [base_colors[int(i)] for i in sxyz[:,0]]


def timeseries_with_drift(n_timepoints, drift_amplitude=0.2):
    t = np.linspace(0, 2*np.pi, n_timepoints)
    drift = np.sin(t) * drift_amplitude
    noise = np.random.randn(n_timepoints) * 0.5
    return drift + noise


f = plt.figure(figsize=(wmax, wmax/3))
f.set_constrained_layout_pads(w_pad=0.2)

grid = (3, 8)
for i in range(3):
    ax = f.add_subplot(*grid, i*grid[1]+1)
    bold_df_smooth = pd.DataFrame({f"Region {i+1}": timeseries_with_drift(n_timepoints) for i in range(n_regions)})
    bold_df_smooth = bold_df_smooth.rolling(window=3, center=True, min_periods=1).mean()
    base_color = np.array(to_rgb(base_colors[i]))
    sns.lineplot(bold_df_smooth,
                palette=[base_color * (0.5 + 0.5 * idx / (n_regions-1)) for idx in range(n_regions)],
                alpha=1.0, linewidth=2, dashes=False, legend=False)
    ax.set(xlabel="Time",ylabel="BOLD",xticks=[],yticks=[],facecolor="lightgray",
           title=("Resting state\ntimeseries" if i==0 else None),
           aspect=(1/ax.get_data_ratio()))


for i in range(3):
    ax = f.add_subplot(*grid, i*grid[1] + 2)
    sns.heatmap(FC[:200, :200], cmap="bone", ax=ax, cbar=False, square=True)
    ax.set(xlabel="vertex", ylabel="vertex", xticks=[], yticks=[], facecolor="lightgray",
           title=("Functional\nconnectivity" if i==0 else None))

for i in range(3):
    for j in range(grid[1]-5):
        ax = f.add_subplot(*grid, i*grid[1] + 3 + j, title=(f"Gradient {j+1}" if i==0 else None))
        ax.set_axis_off()
    

ax = plt.subplot2grid(grid, (0, grid[1]-3), rowspan=3, colspan=3, projection="3d", aspect="equal")
ax.scatter(sxyz[:, 1], sxyz[:, 2], sxyz[:, 3], c=cmark, s=4)
ax.set(xticks=[], yticks =[], zticks = [], title="Gradient space")
_ = ax.set_xlabel("Gradient 1", labelpad=-10)
_ = ax.set_ylabel("Gradient 2", labelpad=-10)
_ = ax.set_zlabel("Gradient 3", labelpad=-10)
ax.view_init(25,-1)

handles = [mpl.patches.Patch(color=base_colors[int(i)]) for i in range(3)]
lgd = f.legend(handles=handles, labels=["001", "002", "N"], title="Participant", ncol=3,
               loc="lower center", bbox_to_anchor=(0.5, 1))
ax.set_box_aspect(ax.get_box_aspect(), zoom=1.2)

fname = f"{figpath}/1A.png"
f.savefig(fname, facecolor="none", bbox_inches="tight")
print(f"Saved figure to {fname}")
plt.close()

In [ ]:
# FIGURE 1B

from matplotlib.colors import to_rgb

base_colors = np.array([cmap_subj(0), cmap_subj(0.5), cmap_subj(.9)])#

sidx = [0,4,5]
sxyz = np.vstack([gembed[s,:,:] for s in sidx])
sxyz = np.hstack([np.repeat(np.arange(3), gembed.shape[1]).reshape(-1,1), sxyz])
mix = np.random.randint(0, len(sxyz), len(sxyz))
sxyz = sxyz[mix]
cmark = [base_colors[int(i)] for i in sxyz[:,0]]


f = plt.figure(figsize=(wmax, wmax/3))
f.set_constrained_layout_pads(w_pad=0.2)

grid = (3, 8)
for i in range(3):
    ax = f.add_subplot(*grid, i*grid[1]+1, aspect=1, title=("Individual\nsurfaces" if i==0 else None))
    circle = plt.Circle((0.5, 0.5), 0.49, transform=ax.transAxes, color=base_colors[i])
    ax.add_patch(circle)
    ax.set_axis_off()


for i in range(3):
    subj = vu.subject(train.subj_list[sidx[i]], train.id)
    D = subj.load_gdist_matrix(h)[11:211, 11:211]
    ax = f.add_subplot(*grid, i*grid[1] + 2)
    sns.heatmap(D[:200, :200], cmap="bone", ax=ax, cbar=False, square=True)
    ax.set(xlabel="vertex", ylabel="vertex", xticks=[], yticks=[], facecolor="lightgray",
           title=("Geodesic\ndistance" if i==0 else None))

for i in range(3):
    for j in range(grid[1]-5):
        ax = f.add_subplot(*grid, i*grid[1] + 3 + j, title=(f"Component {j+1}" if i==0 else None))
        ax.set_axis_off()
    

ax = plt.subplot2grid(grid, (0, grid[1]-3), rowspan=3, colspan=3, projection="3d", aspect="equal")
ax.scatter(sxyz[:, 3], -sxyz[:, 1], sxyz[:, 2], c=cmark, s=4)
ax.set(xticks=[], yticks =[], zticks = [], title="Shared geometry\nembedding")
_ = ax.set_xlabel("Component 3", labelpad=-10)
_ = ax.set_ylabel("Component 1", labelpad=-10)
_ = ax.set_zlabel("Component 2", labelpad=-10)
ax.view_init(25,-1)

handles = [mpl.patches.Patch(color=base_colors[int(i)]) for i in range(3)]
lgd = f.legend(handles=handles, labels=["001", "002", "N"], title="Participant", ncol=3,
               loc="lower center", bbox_to_anchor=(0.5, 1))
ax.set_box_aspect(ax.get_box_aspect(), zoom=1.2)

fname = f"{figpath}/1B.png"
f.savefig(fname, facecolor="none", bbox_inches="tight")
print(f"Saved figure to {fname}")
plt.close()

In [ ]:
# FIGURE 1C

from variograd_utils import load_hdf5
ID = 3900
grp = "train"
srchlight_train_id = np.load(data.outpath(f"{data.id}.{h}.SL_IDs.JE_cauchy{s}_l{l}.npy"))
srchlight_id, srchlight_n = np.unique(srchlight_train_id, return_counts=True)
bins = np.load(data.outpath(f"{data.id}.{h}.SL_bins.JE_cauchy{s}_l{l}.npy"), allow_pickle=True)
nbins = [len(b) - 1 for b in bins]
bin_idx = np.unravel_index(ID, nbins)
axlims = [(edges[i], edges[i+1]) for i, edges in zip(bin_idx, bins)]
ID = srchlight_id[np.argsort(-srchlight_n)][21]
ID = 978

fname = dataset("train").outpath("LKresults.S{0}.G{1}/side{2}/L/LKresults_hdf5_G{1}_sl978.h5")
SLres = load_hdf5(fname.format(s, g+1, l, ID))

SLdf = pd.DataFrame({
    "subject": SLres[grp]["subject_indices"],
    "vertex": SLres[grp]["vertex_indices"],
})

SLdf[["subject", "vertex"]]  -= 1
SLdf["gradient"] = grads[SLdf["subject"], SLdf["vertex"]]
SLdf["centered_gradient"] = SLres[grp]["centered"]["observed"]
SLdf["centered_prediction"] =  SLres[grp]["centered"]["predicted"]
SLdf["subject_mean"] = SLdf[["subject", "gradient"]].groupby("subject").transform("mean")
SLdf[["x", "y", "z"]] = gembed[SLdf["subject"], SLdf["vertex"]]

figsize = (wmax/4.5, wmax/4.5)

In [ ]:
import matplotlib as mpl

cdata = [SLdf["gradient"].copy(), SLdf["subject_mean"].copy(), SLdf["centered_gradient"].copy()]
cmap = [plt.cm.jet, plt.cm.bone, plt.cm.PuOr]
clabel = ["Gradient 1", "Gradient baseline", "Centered gradient"]
x = SLdf["z"]
y = -SLdf["x"]
z = SLdf["y"]


f = plt.figure(figsize=(figsize[0]*2.1, figsize[0]))
f.set_constrained_layout_pads(w_pad=-0.3)

for i in range(3):
    maxval = np.abs(np.percentile(cdata[i], [5, 95])).max()
    norm = mpl.colors.Normalize(vmin=-maxval, vmax=maxval)
    c = cmap[i](norm(cdata[i]))

    nrow = 1 if i!=1 else 2
    ax = plt.subplot2grid((nrow,3), (0,i), rowspan=1, colspan=1, projection="3d")
    p = ax.scatter(z, x, y, c=c, s=10, alpha=1, edgecolor=".3", lw=0.05)

    ax.set(aspect="equal", xticks=[],  yticks=[],  zticks=[],facecolor="none")
    if i != 1:
        _ = ax.set_xlabel("Comp. 3", labelpad=-10)
        _ = ax.set_ylabel("Comp. 1", labelpad=-10)
        _ = ax.set_zlabel("Comp. 2", labelpad=-10)

    cbar = f.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap[i]), ax=ax,
            shrink=.5, pad=0, label=clabel[i], location="top", fraction=0.05)
    cbar.set_label(label=clabel[i])

    box_color = "none"
    ax.set_box_aspect(ax.get_box_aspect(), zoom=0.8)
    ax.xaxis.set_pane_color(box_color)  # RGBA tuple for X pane
    ax.yaxis.set_pane_color(box_color)  # RGBA tuple for Y pane
    ax.zaxis.set_pane_color(box_color)  # RGBA tuple for Z pane

    edge_color = poppy
    ax.set_xticks([*ax.get_xlim()], labels=[])
    ax.set_yticks([*ax.get_ylim()], labels=[])
    ax.set_zticks([*ax.get_zlim()], labels=[])
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo['grid']['color'] = edge_color
        axis._axinfo['grid']['linewidth'] = 3
        axis.line.set_color(poppy)
    ax.set_box_aspect(ax.get_box_aspect(), zoom=1)


fileout = f"{figpath}/1C_gradient-centering.png"
print(f"Saving figure to {fileout}")
f.savefig(fileout, facecolor="none", bbox_inches="tight")
plt.close()

In [ ]:

import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from variograd_utils import dataset

sl_id = 3965

bins = np.load(train.outpath(f"{train.id}.{h}.SL_bins.JE_cauchy{s}_l{l}.npy"), allow_pickle=True)
nbins = [len(b) - 1 for b in bins]
srchlight_train_id = np.load(train.outpath(f"{train.id}.{h}.SL_IDs.JE_cauchy{s}_l{l}.npy"))
srchlight_id, srchlight_n = np.unique(srchlight_train_id, return_counts=True)
to_fill = np.unravel_index(srchlight_id, nbins)
filled = np.zeros(nbins, dtype=bool)
filled[to_fill] = True

xv, yv, zv = np.meshgrid(bins[0], bins[1], bins[2], indexing='ij')

colors = np.zeros(filled.shape + (4,))
colors[to_fill] = [1, 1, 1, 1]
colors[np.unravel_index(sl_id, nbins)] = mpl.colors.hex2color(poppy) + (1,)
edgecolors = np.zeros(filled.shape + (4,))
edgecolors[:, [1,3]] = 1
edgecolors[to_fill] = mpl.colors.hex2color(k) + (1,)
edgecolors[np.unravel_index(sl_id, nbins)] = mpl.colors.hex2color(k) + (1,)

view = (25,-15)
f, ax = plt.subplots(1, 1, subplot_kw={'projection': '3d'},
                     figsize=figsize, facecolor="none", layout="constrained")

ax.voxels(zv, -xv, yv, filled, facecolors=colors, edgecolors=edgecolors, linewidth=.5)

ax.set(aspect="equal", facecolor="none", xticks=[], yticks=[], zticks=[])
ax.view_init(*view)
ax.set_box_aspect(ax.get_box_aspect(), zoom=1.4)
ax.axis("off")

handles = [mpl.patches.Patch(color=poppy, label="example searchlight", alpha=1)]
f.legend(handles=handles, frameon=False, loc="upper center")

fname = f"{figpath}/1C_searchlights.png"
f.savefig(fname, bbox_inches="tight")
print(f"Saved figure to {fname}")
plt.close()